In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1) Load data
# -----------------------------
df = pd.read_csv('data/demo_data/cats/cat_breeds.csv')

# Clean column names (safety)
df.columns = [c.strip().replace(' ', '_').replace('\u00a0','') for c in df.columns]

# Identify Ragdoll row (covers "Ragdoll Cats")
rag = df[df['name'].str.contains('Ragdoll', case=False)].copy()
assert len(rag) == 1, f"Expected one Ragdoll row, found {len(rag)}"

# -----------------------------
# 2) Define metric groups
# -----------------------------
rating_cols = [
    'family_friendly','shedding','general_health','playfulness',
    'children_friendly','grooming','intelligence','other_pets_friendly'
]
quant_cols = ['min_life_expectancy','max_life_expectancy','min_weight','max_weight']

# -----------------------------
# 3) Dataset-level summaries
# -----------------------------
summary = {}
for col in rating_cols + quant_cols:
    summary[col] = {
        'mean': float(df[col].mean()),
        'std': float(df[col].std(ddof=1)),
        'median': float(df[col].median()),
        'p25': float(df[col].quantile(0.25)),
        'p75': float(df[col].quantile(0.75)),
    }

# -----------------------------
# 4) Ragdoll values, percentiles, z-scores
# -----------------------------
rag_vals = rag.iloc[0][rating_cols + quant_cols].to_dict()
z_scores, percentiles = {}, {}

for col in rating_cols + quant_cols:
    mu = summary[col]['mean']
    sd = summary[col]['std']
    val = rag_vals[col]
    z_scores[col] = (val - mu) / sd if sd and sd == sd else np.nan  # guard NaN/zero
    # empirical percentile relative to dataset
    percentiles[col] = float((df[col] <= val).mean())

# -----------------------------
# 5) Comparison table
# -----------------------------
rows = []
for col in rating_cols + quant_cols:
    rows.append({
        'metric': col,
        'ragdoll': rag_vals[col],
        'dataset_mean': round(summary[col]['mean'], 2),
        'dataset_median': round(summary[col]['median'], 2),
        'percentile': round(percentiles[col] * 100, 1),
        'z_score': round(z_scores[col], 2),
    })
comp_df = pd.DataFrame(rows)
comp_df.to_csv('ragdoll_comparison.csv', index=False)

# -----------------------------
# 6) Visualization A: Radar chart (1–5 scale ratings)
# -----------------------------
radar_metrics = rating_cols
values = [rag_vals[m] for m in radar_metrics]
means = [summary[m]['mean'] for m in radar_metrics]
labels = [m.replace('_',' ').title() for m in radar_metrics]

N = len(radar_metrics)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # loop back
val_plot = values + values[:1]
mean_plot = means + means[:1]

plt.figure(figsize=(7, 7))
ax = plt.subplot(111, polar=True)
plt.xticks(angles[:-1], labels)
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1","2","3","4","5"], color="grey", size=8)
plt.ylim(0, 5)
ax.plot(angles, mean_plot, linewidth=1, linestyle='dashed', label='Average (all breeds)')
ax.fill(angles, mean_plot, alpha=0.05)
ax.plot(angles, val_plot, linewidth=2, label='Ragdoll')
ax.fill(angles, val_plot, alpha=0.25)
plt.title('Ragdoll vs. Average — Temperament & Care (1–5)')
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('ragdoll_radar.png', dpi=200)
plt.close()

# -----------------------------
# 7) Visualization B: Life expectancy & weight vs. average
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Life expectancy
life_metrics = ['min_life_expectancy', 'max_life_expectancy']
axes[0].bar(['Ragdoll min','Ragdoll max'],
            [rag_vals[life_metrics[0]], rag_vals[life_metrics[1]]],
            color='#4C78A8', alpha=0.85)
axes[0].hlines([summary[life_metrics[0]]['mean'], summary[life_metrics[1]]['mean']],
               xmin=-0.5, xmax=1.5, colors=['#F58518','#F58518'],
               linestyles='dashed', label='Dataset mean')
axes[0].set_title('Life expectancy (years)')
axes[0].legend()

# Weight
wt_metrics = ['min_weight', 'max_weight']
axes[1].bar(['Ragdoll min','Ragdoll max'],
            [rag_vals[wt_metrics[0]], rag_vals[wt_metrics[1]]],
            color='#72B7B2', alpha=0.85)
axes[1].hlines([summary[wt_metrics[0]]['mean'], summary[wt_metrics[1]]['mean']],
               xmin=-0.5, xmax=1.5, colors=['#F58518','#F58518'],
               linestyles='dashed', label='Dataset mean')
axes[1].set_title('Weight (kg)')
axes[1].legend()

for ax in axes:
    ax.grid(axis='y', alpha=0.2)

plt.suptitle('Ragdoll — Life expectancy and weight vs. dataset average')
plt.tight_layout()
plt.savefig('ragdoll_life_weight.png', dpi=200)
plt.close()

# -----------------------------
# 8) Quick textual takeaways
# -----------------------------
strengths, weaknesses = [], []

# Strengths if at/above 75th percentile (or favorable)
for m in rating_cols:
    if percentiles[m] >= 0.75:
        strengths.append(m)
    elif percentiles[m] <= 0.25:
        weaknesses.append(m)

# Favorable longevity
if percentiles['max_life_expectancy'] >= 0.6:
    strengths.append('max_life_expectancy')

# Grooming: low score implies higher maintenance
if rag_vals['grooming'] <= 2:
    weaknesses.append('grooming (low score ⇒ higher grooming needs)')

print("Generated files: ragdoll_radar.png, ragdoll_life_weight.png, ragdoll_comparison.csv")
print("\nTop strengths (percentile ≥75% or favorable):", strengths)
print("Notable weaker/maintenance areas:", weaknesses)

# Optional: display the first rows of the comparison
print("\nComparison (first rows):")
print(comp_df.head(12))

Generated files: ragdoll_radar.png, ragdoll_life_weight.png, ragdoll_comparison.csv

Top strengths (percentile ≥75% or favorable): ['family_friendly', 'shedding', 'playfulness', 'children_friendly', 'intelligence', 'other_pets_friendly', 'max_life_expectancy']
Notable weaker/maintenance areas: ['grooming', 'grooming (low score ⇒ higher grooming needs)']

Comparison (first rows):
                 metric  ragdoll  dataset_mean  dataset_median  percentile  \
0       family_friendly      5.0          4.31             4.0       100.0   
1              shedding      4.0          2.47             2.0        98.4   
2        general_health      3.0          3.31             3.0        58.1   
3           playfulness      4.0          4.00             4.0        77.4   
4     children_friendly      5.0          4.23             4.0       100.0   
5              grooming      1.0          3.44             4.0         1.6   
6          intelligence      4.0          4.13             4.0        79